# Demand-price coupling under four carbon-tax pathways — fixed SAF volume

The reports hold traffic exogenous and say so explicitly, citing Destination 2050 (about -16 %
demand by 2050) and a national roadmap (-14 %), before placing the question out of scope. Omitting a
feedback because it is hard to estimate assigns it the value zero, which is the one value certain to
be wrong. The point bites here because the instruments delivering the abatement are the same
instruments raising the cost of flying: each lever is scored against a traffic volume it helps
prevent.

AeroMAPS closes that loop: the carbon price enters airline costs, costs reach the ticket price, and
a price-elastic demand model responds. The loop is solved as a fixed point by the model's MDA chain.

**This notebook uses **quantity mandates**, the published SAF volumes held fixed.** If demand falls, the mandated volume does not, so the blend share rises on its own and can saturate at 100 % of drop-in fuel.

The companion notebook `ssp_comparison_share.ipynb` runs the other reading. Both are kept because *Waypoint 2050*
reports SAF as a 2050 volume without saying what would happen to it under lower traffic, and the two
readings give materially different answers once demand responds to price.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

from aeromaps import assemble_processes
from get_data import get_ar6_input_data
import ssp_runs

ar6_data, ar6_years = get_ar6_input_data(start_year=2010, end_year=2100, plot_data=True)

for pathway in ssp_runs.PATHWAYS:
    price_2050 = dict(zip(ar6_years, ar6_data["carbon_tax"][pathway]))[2050]
    print(f"{pathway}: carbon price in 2050 = {price_2050:8.1f} US$2010/tCO2")

## Run the four pathways

In [ ]:
MANDATE = "quantity"

processes = ssp_runs.run_all(ar6_data, ar6_years, mandate=MANDATE)
assembly = assemble_processes(processes)

## Demand response

`rpk_no_elasticity` is the trajectory the reports assume. The gap to `rpk` is the demand the
decarbonisation effort prevents, and which the exogenous-demand construction credits the levers
with carrying.

In [ ]:
summary = ssp_runs.summarise(processes, MANDATE)
display(summary.round(2))

In [ ]:
assembly.plot("rpk_comparison")

In [ ]:
assembly.plot("co2_emissions_comparison")

The net energy cost per RPK is the quantity that actually drives the demand loop. (The feedback
cost bundle replaces `PassengerAircraftSimpleAirfare` with `PassengerAircraftMarginalCost`, so
`airfare_per_ask` is not produced in this configuration.)

In [ ]:
assembly.plot("doc_net_energy_per_rpk_comparison")

### Per-pathway cost breakdown

The comparison above is the net figure; the composition behind it -- how much of the per-RPK energy cost is base fuel price versus carbon tax -- differs by pathway in a way the net number alone does not show. This needs the live process (the carrier-by-carrier breakdown is not reconstructible from committed JSON alone), so the figure is saved here for the document to embed as a static image.

In [ ]:
fig, axes = plt.subplots(
    1, len(processes), figsize=(5.2 * len(processes), 4.2), sharey=True, layout="constrained"
)
for ax, (pathway, process) in zip(axes, processes.items()):
    process.plot("doc_net_energy_per_rpk_breakdown", fig=fig, ax=ax)
    ax.set_title(pathway)
fig.suptitle(f"Energy DOC per RPK breakdown, fixed SAF volume ({MANDATE})")
plt.show()

## Climate consequence

Lower demand lowers every emission term at once, including the non-CO2 ones that offsets cannot
touch.

In [ ]:
assembly.plot("temperature_decomposition_comparison")

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals(), force_cleanup=False)